In [1]:
import pandas as pd
import json
import datasets
from pathlib import Path
from tqdm import tqdm

/home/samoed/Desktop/dialogmteb/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import re
from typing import Any, Dict, List, Union


def parse_semantic(input_str: str) -> Dict[str, Any]:
    """
    Parse a semantic representation string like:
        Send_digital_object ( attachment « screenshot » recipient Personal_contact ( person « Mike » ) )
    
    into a structured Python dict like:
        {
            "target": "Send_digital_object",
            "attachment": "screenshot",
            "Personal_contacts": ["Mike"]
        }

    Rules:
    - Top-level function name -> "target"
    - slot « value » -> {slot: value}
    - Personal_contact(person « X ») -> append person into "Personal_contacts" list
    - Electronic_message(topic « Y ») -> {"Electronic_message": Y}
    """
    input_str = input_str.strip()
    
    # Extract target and arguments
    match = re.match(r"(\w+)\s*\((.*)\)", input_str)
    if not match:
        # Handle case like "Create_note ( app « Google Keep » )"
        match = re.match(r"(\w+)\s*\((.*)\)", input_str)
    if not match:
        raise ValueError(f"Invalid format: {input_str}")
    
    target, inside = match.groups()
    result: Dict[str, Any] = {"target": target}

    # Tokenize by slots, handling nested functions carefully
    tokens = split_tokens(inside)

    for token in tokens:
        token = token.strip()
        # slot « value »
        if "«" in token and "»" in token and "(" not in token:
            slot, val = re.match(r"(\w+)\s*«\s*(.*?)\s*»", token).groups()
            result[slot] = val

        # Personal_contact ( person « X » )
        elif token.startswith("Personal_contact"):
            sub_match = re.search(r"person\s*«\s*(.*?)\s*»", token)
            if sub_match:
                person = sub_match.group(1)
                result.setdefault("Personal_contacts", []).append(person)

        # Electronic_message ( topic « Y » )
        elif token.startswith("Electronic_message"):
            sub_match = re.search(r"topic\s*«\s*(.*?)\s*»", token)
            if sub_match:
                result["Electronic_message"] = sub_match.group(1)

        else:
            # Fallback: try slot « value » again
            sub_match = re.match(r"(\w+)\s*«\s*(.*?)\s*»", token)
            if sub_match:
                slot, val = sub_match.groups()
                result[slot] = val

    return result


def split_tokens(s: str) -> List[str]:
    """
    Split inside (...) into tokens, respecting nested parentheses.
    Example:
        "attachment « screenshot » recipient Personal_contact ( person « Mike » )"
    -> ["attachment « screenshot »", "recipient Personal_contact ( person « Mike » )"]
    """
    tokens: List[str] = []
    buf = []
    depth = 0
    for ch in s:
        if ch == "(":
            depth += 1
            buf.append(ch)
        elif ch == ")":
            depth -= 1
            buf.append(ch)
        elif ch == " " and depth == 0:
            # Split on spaces between tokens
            if buf:
                tokens.append("".join(buf))
                buf = []
        else:
            buf.append(ch)
    if buf:
        tokens.append("".join(buf))
    return merge_tokens(tokens)


def merge_tokens(tokens: List[str]) -> List[str]:
    """
    Merge tokens like ["recipient", "Personal_contact(...)", "attachment", "« screenshot »"]
    into ["recipient Personal_contact(...)", "attachment « screenshot »"]
    """
    merged: List[str] = []
    buf: List[str] = []
    for tok in tokens:
        buf.append(tok)
        if "»" in tok or tok.endswith(")"):
            merged.append(" ".join(buf))
            buf = []
    if buf:
        merged.append(" ".join(buf))
    return merged


if __name__ == "__main__":
    examples = [
        "Send_digital_object ( attachment « screenshot » cc Personal_contact ( person « Josh » ) recipient Personal_contact ( person « Mike » ) )",
        "Cancel_ride ( dropoff_location « Walmart » provider « Lyft » )",
        "Send_digital_object ( object « ETA » provider « whatsapp » )",
        "Create_note ( app « Notas » trigger_time « 6 pm » )",
        "Create_note ( app « Google Keep » )",
        "Send_digital_object ( medium « text message » object Electronic_message ( topic « party plans » ) recipient Personal_contact ( person « Kristin » ) )"
    ]

    for ex in examples:
        print(ex)
        print(parse_semantic(ex))
        print("-" * 60)


Send_digital_object ( attachment « screenshot » cc Personal_contact ( person « Josh » ) recipient Personal_contact ( person « Mike » ) )
{'target': 'Send_digital_object', 'attachment': 'screenshot'}
------------------------------------------------------------
Cancel_ride ( dropoff_location « Walmart » provider « Lyft » )
{'target': 'Cancel_ride', 'dropoff_location': 'Walmart', 'provider': 'Lyft'}
------------------------------------------------------------
Send_digital_object ( object « ETA » provider « whatsapp » )
{'target': 'Send_digital_object', 'object': 'ETA', 'provider': 'whatsapp'}
------------------------------------------------------------
Create_note ( app « Notas » trigger_time « 6 pm » )
{'target': 'Create_note', 'app': 'Notas', 'trigger_time': '6 pm'}
------------------------------------------------------------
Create_note ( app « Google Keep » )
{'target': 'Create_note', 'app': 'Google Keep'}
------------------------------------------------------------
Send_digital_objec

In [3]:
from dataclasses import dataclass
from typing import Any, Optional


@dataclass(frozen=True)
class Token:
    """Lexical token with a simple type and optional value."""
    type: str
    value: Optional[str] = None  # IDENT | VALUE | '(' | ')'


def lex(s: str) -> list[Token]:
    """
    Convert the input string into tokens:
      - IDENT: letters, digits, underscore (e.g., Send_digital_object, recipient)
      - VALUE: text inside guillemets « ... »
      - LPAREN: '('
      - RPAREN: ')'
    Whitespace is ignored.
    """
    tokens: list[Token] = []
    i, n = 0, len(s)
    while i < n:
        ch = s[i]
        if ch.isspace():
            i += 1
            continue
        if ch == '(':
            tokens.append(Token('LPAREN', '('))
            i += 1
            continue
        if ch == ')':
            tokens.append(Token('RPAREN', ')'))
            i += 1
            continue
        if ch == '«':
            # Read until the matching guillemet »
            j = i + 1
            buf: list[str] = []
            while j < n and s[j] != '»':
                buf.append(s[j])
                j += 1
            if j >= n:
                raise ValueError("Unclosed guillemet value starting at position "
                                 f"{i}: expected '»'.")
            tokens.append(Token('VALUE', ''.join(buf).strip()))
            i = j + 1
            continue
        if ch.isalnum() or ch == '_':
            j = i
            while j < n and (s[j].isalnum() or s[j] == '_'):
                j += 1
            ident = s[i:j]
            tokens.append(Token('IDENT', ident))
            i = j
            continue
        # Ignore any other single punctuation characters
        i += 1
    return tokens


class Parser:
    """
    Recursive-descent parser for expressions like:
      Send_digital_object ( attachment « screenshot » recipient Personal_contact ( person « Mike » ) )
    Produces a simple AST, then we transform the AST into the desired dict.
    """

    def __init__(self, tokens: list[Token]) -> None:
        self.toks = tokens
        self.pos = 0

    def _peek(self, k: int = 0) -> Optional[Token]:
        idx = self.pos + k
        return self.toks[idx] if 0 <= idx < len(self.toks) else None

    def _consume(self, ttype: str | None = None) -> Token:
        tok = self._peek()
        if tok is None:
            raise ValueError("Unexpected end of input.")
        if ttype is not None and tok.type != ttype:
            raise ValueError(f"Expected token {ttype}, found {tok.type} ({tok.value}).")
        self.pos += 1
        return tok

    # ---------- Grammar ----------
    # FuncCall := IDENT '(' Args? ')'
    # Args     := Arg+
    # Arg      := IDENT VALUE                    # slot « value »
    #          |  IDENT IDENT '(' Args? ')'      # slot + NestedFunction
    #          |  IDENT '(' Args? ')'            # (rare) bare NestedFunction

    def parse_function(self) -> dict[str, Any]:
        name_tok = self._consume('IDENT')
        self._consume('LPAREN')
        args: list[Any] = []
        while (t := self._peek()) is not None and t.type != 'RPAREN':
            args.append(self.parse_arg())
        self._consume('RPAREN')
        return {"type": "func", "name": name_tok.value, "args": args}

    def parse_function_with_name(self, fname: str) -> dict[str, Any]:
        self._consume('LPAREN')
        args: list[Any] = []
        while (t := self._peek()) is not None and t.type != 'RPAREN':
            args.append(self.parse_arg())
        self._consume('RPAREN')
        return {"type": "func", "name": fname, "args": args}

    def parse_arg(self) -> dict[str, Any]:
        # IDENT VALUE  -> slot-value
        if (a := self._peek()) and a.type == 'IDENT' and \
        (b := self._peek(1)) and b.type == 'VALUE':
            slot = self._consume('IDENT').value  # type: ignore[assignment]
            val = self._consume('VALUE').value   # type: ignore[assignment]
            return {"type": "slot", "slot": slot, "value": val}

        # IDENT IDENT '(' ... ')' -> slot + nested function
        if (a := self._peek()) and a.type == 'IDENT' and \
        (b := self._peek(1)) and b.type == 'IDENT' and \
        (c := self._peek(2)) and c.type == 'LPAREN':
            slot = self._consume('IDENT').value  # type: ignore[assignment]
            fname = self._consume('IDENT').value  # type: ignore[assignment]
            func = self.parse_function_with_name(fname)
            return {"type": "slot_func", "slot": slot, "func": func}

        # IDENT '(' ... ')' -> bare nested function (no slot name)
        if (a := self._peek()) and a.type == 'IDENT' and \
        (b := self._peek(1)) and b.type == 'LPAREN':
            func = self.parse_function()
            return {"type": "func_arg", "func": func}

        # NEW RULE: skip unsupported IDENT IDENT (like "device InferFromContext")
        if (a := self._peek()) and a.type == 'IDENT' and \
        (b := self._peek(1)) and b.type == 'IDENT':
            # consume both and ignore
            self._consume('IDENT')
            self._consume('IDENT')
            return {"type": "skip"}

        # Otherwise: ignore unexpected
        tok = self._consume()
        return {"type": "skip", "value": tok.value}

# ---------- AST -> dict transformer ----------

def _collect_slot(args: list[dict[str, Any]], slot_name: str) -> list[str]:
    """Helper: collect all slot values with the given slot_name from a function's args."""
    vals: list[str] = []
    for a in args:
        if a["type"] == "slot" and a["slot"] == slot_name:
            vals.append(a["value"])
    return vals


def ast_to_output(ast: dict[str, Any]) -> dict[str, Any]:
    """
    Transform the AST into the desired flattened JSON-like dict:
      - Top-level function name -> "target"
      - slot « value » -> direct key/value
      - slot Personal_contact(...) -> add inside "Personal_contacts" list (extract 'person')
      - slot Electronic_message(...) -> "Electronic_message": topic
    Unknown nested functions become a dict under their function name with their slot-value pairs.
    """
    if ast.get("type") != "func":
        raise ValueError("AST must start with a function node.")
    out: dict[str, Any] = {"target": ast["name"]}

    for arg in ast["args"]:
        if arg["type"] == "slot":
            out[arg["slot"]] = arg["value"]
            continue

        if arg["type"] == "skip":
            continue

        if arg["type"] in ("slot_func", "func_arg"):
            func = arg["func"] if arg["type"] == "slot_func" else arg["func"]
            fname: str = func["name"]
            fargs: list[dict[str, Any]] = func["args"]

            # Specialization: Personal_contact(person « X »)
            if fname == "Personal_contact":
                people = _collect_slot(fargs, "person")
                if people:
                    out.setdefault("Personal_contacts", [])
                    # type: ignore[index]
                    out["Personal_contacts"].extend(people)  # type: ignore[operator]
                continue

            # Specialization: Electronic_message(topic « Y »)
            if fname == "Electronic_message":
                topics = _collect_slot(fargs, "topic")
                if topics:
                    out["Electronic_message"] = topics[-1]
                continue

            # Generic fallback: store nested function as a dict of its slots
            nested: dict[str, Any] = {}
            for fa in fargs:
                if fa["type"] == "slot":
                    nested[fa["slot"]] = fa["value"]
            if nested:
                out[fname] = nested
            continue

        # Safety fallback (shouldn't happen)
        raise ValueError(f"Unknown arg node type: {arg['type']}")

    return out


def parse_semantic(s: str) -> dict[str, Any]:
    """
    Public API: parse a semantic string into the target dict.
    This function:
      1) Lexes the string into tokens,
      2) Builds an AST with a recursive-descent parser,
      3) Transforms the AST into the desired JSON-like dict.
    """
    tokens = lex(s.strip())
    ast = Parser(tokens).parse_function()
    return ast_to_output(ast)

In [4]:
examples: list[tuple[str, dict[str, Any]]] = [
    (
        "Send_digital_object ( attachment « screenshot » cc Personal_contact ( person « Josh » ) recipient Personal_contact ( person « Mike » ) )",
        {
            "target": "Send_digital_object",
            "attachment": "screenshot",
            "Personal_contacts": ["Josh", "Mike"],
        },
    ),
    (
        "Send_digital_object ( medium « text message » object Electronic_message ( topic « party plans » ) recipient Personal_contact ( person « Kristin » ) )",
        {
            "target": "Send_digital_object",
            "medium": "text message",
            "Electronic_message": "party plans",
            "Personal_contacts": ["Kristin"],
        },
    ),
    (
        "Cancel_ride ( dropoff_location « Walmart » provider « Lyft » )",
        {
            "target": "Cancel_ride",
            "dropoff_location": "Walmart",
            "provider": "Lyft",
        },
    ),
    (
        "Send_digital_object ( object « ETA » provider « whatsapp » )",
        {
            "target": "Send_digital_object",
            "object": "ETA",
            "provider": "whatsapp",
        },
    ),
    (
        "Create_note ( app « Notas » trigger_time « 6 pm » )",
        {
            "target": "Create_note",
            "app": "Notas",
            "trigger_time": "6 pm",
        },
    ),
    (
        "Create_note ( app « Google Keep » )",
        {
            "target": "Create_note",
            "app": "Google Keep",
        },
    ),
]

for inp, expected in examples:
    got = parse_semantic(inp)
    print(inp)
    print("=>", got)
    assert got == expected, f"Mismatch!\nExpected: {expected}\nGot: {got}"
    print("-" * 60)

Send_digital_object ( attachment « screenshot » cc Personal_contact ( person « Josh » ) recipient Personal_contact ( person « Mike » ) )
=> {'target': 'Send_digital_object', 'attachment': 'screenshot', 'Personal_contacts': ['Josh', 'Mike']}
------------------------------------------------------------
Send_digital_object ( medium « text message » object Electronic_message ( topic « party plans » ) recipient Personal_contact ( person « Kristin » ) )
=> {'target': 'Send_digital_object', 'medium': 'text message', 'Electronic_message': 'party plans', 'Personal_contacts': ['Kristin']}
------------------------------------------------------------
Cancel_ride ( dropoff_location « Walmart » provider « Lyft » )
=> {'target': 'Cancel_ride', 'dropoff_location': 'Walmart', 'provider': 'Lyft'}
------------------------------------------------------------
Send_digital_object ( object « ETA » provider « whatsapp » )
=> {'target': 'Send_digital_object', 'object': 'ETA', 'provider': 'whatsapp'}
----------

In [5]:
folder = Path("/home/samoed/Downloads/presto_v1/test_partitions/en-US/")

In [6]:
ff = list(folder.glob("./*/*.jsonl"))[2:]
ff

[PosixPath('/home/samoed/Downloads/presto_v1/test_partitions/en-US/en-US_non_contextual/test.jsonl'),
 PosixPath('/home/samoed/Downloads/presto_v1/test_partitions/en-US/en-US_no_phenomena/test.jsonl'),
 PosixPath('/home/samoed/Downloads/presto_v1/test_partitions/en-US/en-US_revisions/test.jsonl')]

In [7]:
r = list(Path("/home/samoed/Downloads/presto_v1/test_partitions/").glob("./*/*/*.jsonl"))

In [8]:
r

[PosixPath('/home/samoed/Downloads/presto_v1/test_partitions/de-DE/de-DE_code-mixing/test.jsonl'),
 PosixPath('/home/samoed/Downloads/presto_v1/test_partitions/de-DE/de-DE_disfluency/test.jsonl'),
 PosixPath('/home/samoed/Downloads/presto_v1/test_partitions/de-DE/de-DE_non_contextual/test.jsonl'),
 PosixPath('/home/samoed/Downloads/presto_v1/test_partitions/de-DE/de-DE_no_phenomena/test.jsonl'),
 PosixPath('/home/samoed/Downloads/presto_v1/test_partitions/de-DE/de-DE_revisions/test.jsonl'),
 PosixPath('/home/samoed/Downloads/presto_v1/test_partitions/en-US/en-US_code-mixing/test.jsonl'),
 PosixPath('/home/samoed/Downloads/presto_v1/test_partitions/en-US/en-US_disfluency/test.jsonl'),
 PosixPath('/home/samoed/Downloads/presto_v1/test_partitions/en-US/en-US_non_contextual/test.jsonl'),
 PosixPath('/home/samoed/Downloads/presto_v1/test_partitions/en-US/en-US_no_phenomena/test.jsonl'),
 PosixPath('/home/samoed/Downloads/presto_v1/test_partitions/en-US/en-US_revisions/test.jsonl'),
 PosixPa

In [9]:
from datasets import Dataset, DatasetDict, Features, Value, List

def process_ds(data: list)->Dataset:
    all_keys = set()
    for row in data:
        all_keys.update(row.keys())

    # Convert into a dataset, filling missing with None
    normalized_data = [
        {k: row.get(k, None) if k != "dialog" else row[k] for k in all_keys }
        for row in data
    ]
    for row in normalized_data:
        if row["dialog"] == "none":
            print(row["dialog"])
    part_ds = Dataset.from_list(normalized_data)
    return part_ds

def process_ds_dict(data: dict[str, dict]) -> DatasetDict:
    all_keys = set()
    for ds in data.values():
        for row in ds:
            all_keys.update(row.keys())

    features = {
        col: Value("string")
        for col in all_keys
    }
    features["dialog"] = List({"content": Value("string"), "role": Value("string")})
    features = Features(features)

    # Convert into a dataset, filling missing with None
    for name, ds in data.items():
        ds = [
            {k: row.get(k, "none") for k in all_keys}
            for row in ds
        ]
        data[name] = Dataset.from_list(ds, features=features)
    return DatasetDict(data)

In [10]:
for file in r:
    split = []
    subset = file.parent.name
    with file.open() as f:
        for line in f:
            split.append(json.loads(line))
    d = []
    for row in tqdm(split, desc=subset):
        dialog = []
        for replics in row["metadata"]["previous_turns"]:
            dialog.append({
                "role": "user",
                "content": replics["user_query"],
            })
            dialog.append({
                "role": "assistant",
                "content": replics["response_text"],
            })
        dialog.append({
            "role": "user",
            "content": row["inputs"]
        })
        parsed = parse_semantic(row["targets"])
        d.append({
            "dialog": dialog,
            "targets_raw": row["targets"],
            # "metadata": row["metadata"],
            **parsed
        })
    ds = process_ds(d)
    ds.push_to_hub("DeepPavlov/presto", subset, split="test")

Creating parquet from Arrow format: 100%|██████████| 5/5 [00:00<00:00, 220.66ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :  11%|█▏        | 65.4kB /  577kB, 81.8kB/s  







Processing Files (1 / 1)                : 100%|██████████|  577kB /  577kB,  240kB/s  


Processing Files (1 / 1)                : 100%|██████████|  577kB /  577kB,  206kB/s  
New Data Upload                         : 100%|██████████|  511kB /  511kB,  183kB/s  
                                        : 100%|██████████|  577kB /  577kB            
Creating parquet from Arrow format: 100%|██████████| 6/6 [00:00<00:00, 242.10ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :  10%|▉         | 63.8kB /  662kB, 79.8kB/s  
Processing Files (0 / 1)                : 100%|█████████▉|  659kB /  662kB,  659kB/s  







Processing Files (1 / 1)                :

In [17]:
r = Path("/home/samoed/Downloads/presto_v1/").glob("*.jsonl")

In [18]:
from collections import defaultdict

ds = defaultdict(lambda: defaultdict(list))
for file in r:
    if file.name == 'presto_dataset.jsonl':
        continue
    split_name = file.name.split('_')[1].split('.')[0]
    split = []
    with file.open() as f:
        for line in f:
            split.append(json.loads(line))
    for row in tqdm(split, desc=split_name):
        dialog = []
        subset_type = row["metadata"]["linguistic_phenomena"]
        lang = row["metadata"]["locale"]
        subset = f'{lang}_{subset_type}'

        for replics in row["metadata"]["previous_turns"]:
            dialog.append({
                "role": "user",
                "content": replics["user_query"],
            })
            dialog.append({
                "role": "assistant",
                "content": replics["response_text"],
            })
        dialog.append({
            "role": "user",
            "content": row["inputs"]
        })
        parsed = parse_semantic(row["targets"])
        ds[subset][split_name].append({
            "dialog": dialog,
            "targets_raw": row["targets"],
            "metadata": row["metadata"],
            **parsed
        })

train: 100%|██████████| 276259/276259 [00:10<00:00, 27008.21it/s]


In [19]:
for subset, v in tqdm(ds.items(), leave=True, total=len(ds)):
    process_ds_dict(v).push_to_hub("DeepPavlov/presto", subset)

  0%|          | 0/42 [00:00<?, ?it/s]


Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 53.39ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :  58%|█████▊    |  554kB /  957kB,  231kB/s  









Processing Files (1 / 1)                : 100%|██████████|  957kB /  957kB,  217kB/s  

Processing Files (1 / 1)                : 100%|██████████|  957kB /  957kB,  208kB/s  
New Data Upload                         : 100%|██████████|  957kB /  957kB,  208kB/s  
                                        : 100%|██████████|  957kB /  957kB            
Uploading the dataset shards: 100%|██████████| 1/1 [00:05<00:00,  5.77s/ shards]

Creating parquet from Arrow format: 100%|██████████| 5/5 [00:00<00:00, 63.73ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :   3%|▎         | 65.4kB / 2.23MB, 81.7kB/s  
Processing Files (0 / 1)                :  28%|██▊   